> TO DO

1. Inspect whether we can make use of some measure of sensitivity from totalistic CAs (and highlight how a general totalistic CA is different from the case we are studying here now).
2. Make a visual comparison of the jaggedness with some time evolution patterns
3. Find better names than 'jaggedness'. Possibilities: horizontal sensitivity vs. vertical sensitivity; density sensitivity vs. flip sensitivity; neighbourhood sensitivity vs identity sensitivity; peripheral reactivity vs. reflective reactivity.

# Define jaggedness

In this notebook we will present progressively better definitions of "jaggedness" (actual name pending).

This is a measure that allows us to predict the way in which a defect may spread through a network, based on the LLNA's local update rule and the the networks' degree distribution.

## General principles

In the context of LLNAs, introducing a single defect has two effects:
1. The node that changes state will obey a different update rule
2. The nodes in the changed node's neighbourhood will experience a change in their neighbourhood's state density

These changes can have a big effect or a small effect on the dynamics of the system.

## The naive definition

In [ ]:
import torch as tc
import igraph as ig
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import rcParams

# Enable LaTeX and set Times New Roman as the font
rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "text.latex.preamble": r"\usepackage{amsmath}"  # Optional: Use LaTeX packages
})

from tqdm import tqdm

import sys, os
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(parent_dir)

from src.automata import LLNA
from src.simulation import *

%load_ext autoreload
%autoreload 2

In the first definition we only look at the effect of slightly modifying the local density. In particular we look at how many opportunities the LLNA has to change its output when a defect is introduced. We quantify this by counting the number of 'borders' in the diagram of the local update rule, dividing by the total possible number. Some examples are listed below:

In [ ]:
from src.analysis import nbh_sensitivity

resolution = 3
born_if = [0,2]
survive_if = [1]
nbh_sensitivity(resolution, born_if, survive_if)

In [ ]:
SAVEFIG=False

fig, ax = plt.subplots(1,1,figsize=(7,1.7))

resolution = 3
born_if = [1]
survive_if = [0, 2]
model = LLNA(resolution, x=born_if, y=survive_if, iso=True)
model.diagram(ax=ax)
j = nbh_sensitivity(resolution, born_if, survive_if)
borders = j * 2*(resolution-1)
ax.set_title(f"Local update rule diagram for LLNA {model.__str__(latex=True)}.")

fig.tight_layout()
if SAVEFIG:
    # Note: saving as PDF fails to show the hatches
    plt.savefig(f"update-rule-diagram_rule{model.__str__()}.png", dpi=600)

In [ ]:
SAVEFIG=False

fig, axs = plt.subplots(4,1,figsize=(7, 5.5))

resolution = 5
born_if = [0, 1, 2, 3, 4]
survive_if = []
model = LLNA(resolution, x=born_if, y=survive_if)
model.diagram(ax=axs[0])
axs[0].set_title(fr"Impact only on the perturbed node $v_i$")
axs[0].set_xlabel(None)
axs[0].set_ylabel(None)
axs[0].get_legend().remove()

resolution = 5
born_if = [0, 2, 4]
survive_if = [0, 2, 4]
model = LLNA(resolution, x=born_if, y=survive_if)
model.diagram(ax=axs[1])
axs[1].set_title(r"Impact only on the neighbourhood nodes in $\mathcal{N}(v_i)$")
axs[1].set_xlabel(None)
axs[1].set_ylabel(None)
axs[1].get_legend().remove()

resolution = 5
born_if = [0, 2, 4]
survive_if = [1, 3]
model = LLNA(resolution, x=born_if, y=survive_if)
model.diagram(ax=axs[2])
axs[2].set_title(f"Large impact on both the perturbed node and the neighbourhood nodes")
axs[2].set_xlabel(None)
axs[2].set_ylabel(None)
axs[2].get_legend().remove()

resolution = 5
born_if = [0, 1]
survive_if = [0, 1, 2] 
model = LLNA(resolution, x=born_if, y=survive_if)
model.diagram(ax=axs[3])
axs[3].set_title(f"Little impact on the perturbed node and neighbourhood nodes")
axs[3].set_xlabel(fr"State density $\rho_j$ associated with node $v_j$", size=14)
axs[3].set_ylabel(None)

# fig.supxlabel(fr"State density $\rho_j$ associated with node $v_j$")
fig.supylabel(fr"Node becomes ...", size=14)
fig.tight_layout()

if SAVEFIG:
    # Note: saving as PDF fails to show the hatches
    plt.savefig(f"some-examples-of-obvious-diagrams.png", dpi=600)

Note:
1. We do not count the outer borders (those at $\rho = 0$ and $\rho = 1$) because they cannot be crossed.
2. The $B$ set has the same number of edges as the $B^\text{C}$ set, so the representation in the diagram does not matter.
3. There are always $2(R-1)$ possible borders.
4. There are more possibilities with an average number of borders than with a low or high number of borders.

## Add notion of changing the central node

If the central node is perturbed, the neighbourhood density is not altered, but the dynamical outcome may of course change as well. This is expressed by looking at some kind of XOR operation between the survive set and the born set. Again this value is normalised over the maximum (which is simply the resolution).

In [ ]:
from src.analysis import id_sensitivity

resolution = 3
born_if = [0,1,2]
survive_if = []
model = LLNA(resolution, x=born_if, y=survive_if, iso=True)
id_sensitivity(resolution, born_if, survive_if)

In this spirit, the LLNA that can maximally change the outcome is the one that is the most jagged, and for which the born set is the complement of the survive set, for example

In [ ]:
SAVEFIG=False

fig, axs = plt.subplots(2,1,figsize=(7, 3.5))

resolution = 5
born_if = [0, 2, 4]
survive_if = [1, 3]
model = LLNA(resolution, x=born_if, y=survive_if)
model.diagram(ax=axs[0])
j = id_sensitivity(resolution, born_if, survive_if)
flips = int(j*resolution)
axs[0].set_title(f"{model.__str__(latex=True)}: {flips} flip(s) out of {resolution} possibilities: identity sensitivity of {round(j,2)}.")
axs[0].get_legend().remove()
axs[0].set_xlabel(None)
axs[0].set_ylabel(None)

resolution = 7
born_if = [0, 1, 2, 6]
survive_if = [3, 4, 6]
model = LLNA(resolution, x=born_if, y=survive_if)
model.diagram(ax=axs[1])
j = id_sensitivity(resolution, born_if, survive_if)
flips = int(j*resolution)
axs[1].set_title(f"{model.__str__(latex=True)}: {flips} flip(s) out of {resolution} possibilities: identity sensitivity of {round(j,2)}.")
axs[1].set_xlabel(fr"State density $\rho_j$ associated with node $v_j$", size=14)
axs[1].set_ylabel(None)

fig.suptitle("LLNA rule diagram illustrating identity sensitivity")
fig.supylabel("Node becomes ...", size=14)
fig.tight_layout()

It is not a priori clear how we should balance these two effects: the effect of changing the neighbourhood density, and the effect of changing the central node state. Some observations include:
1. Changing the state of the central node will always have the same effect; it is not depedent on the density of the local neighbourhood (and hence not dependent on the node degree)
2. LLNAs for which the 'central node jaggedness' is low and the 'naive jaggedness' is high, will generally not be very affected by introducing a defect. It is not clear, however, how important these definitions are compared to one another.
3. For isomorphic LLNAs, values from both jaggedness definitions are unchanged under an isomorphism

Also, a single jaggedness value is associated with a varying number of LLNAs. That is to say ...

_naive jaggedness definition_
- no borders: only one choice of border placement
- one border: $R-1$ choices
- two borders: $(R-1)(R-2)/2$ choices
- three borders: $(R-1)(R-2)(R-3)/6$ choices
- $N$ borders: $\prod_{i=1}^N(R-i)/i = \binom{R-1}{N}$ choices

Each of these border choices allows for two choices of intervals (either enabled or disabled). Summing all these possiblities, we find
$$
\left(2\sum_{N=0}^{R-1}\binom{R-1}{N}\right)^2 = \left(2 \times 2^{R-1}\right)^2 = 2^{2R},
$$
as required from the LLNA definition.

_central node jaggedness definition_
- zero central node jaggedness: all blocks must be identical: $2^R$ possible choices
- $1/R$ central node jaggedness: all-but-one blocks must be identical: $2^R R$ possible choices
- $2/R$ central node jaggedness: all-but-two blocks must be identical: $2^R R (R-1)/2$ possible choices
- $k/R$ central node jaggedness: all-but-$k$ blocks must be identical: $2^R R!/(R-k)!/k!$ possible choices

This again sums to a total of $2^{2R}$ choices, as required.

There does not seem to be an a priori reason to relate both jaggedness definitions. Let's inspect whether there is some correlation.

In [ ]:
from src.analysis import eca_as_binary

# TODO there is definitely a nicer way to do this! That's not really the point here though
def int_to_set(integer, resolution):
    if integer >= 2**resolution:
        raise Exception(f"The value of the integer encoding the density intervals should not exceed {2**resolution-1}.")
    intervals = list(range(resolution))
    density_set = []
    brp = eca_as_binary(integer)
    for i, char in enumerate(brp[::-1]):
        if char=='1':
            density_set.append(i)
    return np.array(density_set)

Check whether the values of the neighbourhood sensitivity and the identity sensitivity are independent, as they should be. We can check this computationally below, but it also just makes sense.

In [ ]:
resolution = 8 # the higher, the (exponentially) more time it takes to calculate
B_sets = [int_to_set(integer, resolution) for integer in range(2**resolution)]
S_sets = B_sets.copy()

naive_js = []
naive_js_cn = []
for B_set in B_sets:
    for S_set in S_sets:
        naive_js += [nbh_sensitivity(resolution, B_set, S_set)]
        naive_js_cn += [id_sensitivity(resolution, B_set, S_set)]

# the values are independent of each other, indeed!
from scipy.stats import pearsonr
pr = pearsonr(naive_js, naive_js_cn).statistic

print(f"Checked for resolution {resolution}: the Pearson's R is {pr}. There is no correlation.")

In [ ]:
import itertools

# TODO: add a grid with size vs size with colours indicating the average jaggedness.

# Define parameters
resolution = 7
R_set = np.arange(resolution)

B_sizes = range(resolution + 1)
S_sizes = range(resolution + 1)

# Precompute all combinations for given sizes
combinations_by_size = {size: list(itertools.combinations(R_set, size)) for size in range(resolution + 1)}

# Initialize grids for results
j_values_mean_grid = np.zeros((resolution + 1, resolution + 1))
j_values_cn_mean_grid = np.zeros((resolution + 1, resolution + 1))

# Compute mean jaggedness values
for B_size in B_sizes:
    print(f"Working on B_size {B_size}/{resolution}.    ", end='\r')
    B_sets = combinations_by_size[B_size]
    for S_size in S_sizes:
        S_sets = combinations_by_size[S_size]
        # Compute jaggedness values for all combinations
        j_values = [
            nbh_sensitivity(resolution, B_set, S_set)
            for B_set in B_sets for S_set in S_sets
        ]
        j_values_cn = [
            id_sensitivity(resolution, B_set, S_set)
            for B_set in B_sets for S_set in S_sets
        ]
        # Store means in grids
        j_values_mean_grid[B_size, S_size] = np.mean(j_values)
        j_values_cn_mean_grid[B_size, S_size] = np.mean(j_values_cn)


In [ ]:
SAVEFIG=False

fig, axs = plt.subplots(1,2,figsize=(7, 4))

im0 = axs[0].imshow(j_values_cn_mean_grid, cmap='Greens')
im1 = axs[1].imshow(j_values_mean_grid, cmap='Reds')

for ax in axs:
    ax.set_xlabel("B set size", size=14)
    ax.set_ylabel("S set size", size=14)
    ax.set_xticks(range(resolution+1))
axs[0].set_title("Identity sensitivity\n(average)", size=16)
axs[1].set_title("Neighbourhood sensitivity\n(average)", size=16)

# Create a colorbar that aligns perfectly
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Create an axis for the colorbar
divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.05)  # Adjust size and padding
cbar = plt.colorbar(im0, cax=cax)  # Use the new axis for the colorbar
# Same for the second one
divider = make_axes_locatable(axs[1])
cax = divider.append_axes("right", size="5%", pad=0.05)  # Adjust size and padding
cbar = plt.colorbar(im1, cax=cax)  # Use the new axis for the colorbar

fig.tight_layout()

if SAVEFIG:
    # Note: saving as PDF fails to show the hatches
    plt.savefig(f"ID-and-NS-res7.png", dpi=600)

Interestingly, these values seem to convey very different information! Also note that the observed symmetry corresponds with imposed isomorphisms.

The above discussion shows the distribution of the values simply as a result of combinatorics. A complete and useful definition of jaggedness should probably take into account these combinatorial aspects, allowing for some kind of normalisation.

In [ ]:
# exhaustive figure for all resolution-3 models
# TODO: add a grid with size vs size with colours indicating the average jaggedness.

# Define parameters
resolution = 3
betas = range(2**resolution)
sigmas = range(2**resolution)

# Initialize grids for results
j_values_mean_grid = np.zeros((2**resolution, 2**resolution))
j_values_cn_mean_grid = np.zeros((2**resolution, 2**resolution))

# Compute mean jaggedness values
for beta in betas:
    print(f"Working on beta {beta}/{2**resolution-1}.    ", end='\r')
    B_set = []
    for k, digit in enumerate(np.base_repr(beta)[::-1]):
        if digit=='1':
            B_set.append(k)
    for sigma in sigmas:
        S_set = []
        for k, digit in enumerate(np.base_repr(sigma)[::-1]):
            if digit=='1':
                S_set.append(k)
        j_value = nbh_sensitivity(resolution, B_set, S_set)
        j_value_cn = id_sensitivity(resolution, B_set, S_set)
        # Store means in grids
        j_values_mean_grid[beta, sigma] = j_value
        j_values_cn_mean_grid[beta, sigma] = j_value_cn

In [ ]:
SAVEFIG=False

fig, axs = plt.subplots(1,2,figsize=(7, 4))

im0 = axs[0].imshow(j_values_cn_mean_grid, cmap='Greens')
im1 = axs[1].imshow(j_values_mean_grid, cmap='Reds')

for ax in axs:
    ax.set_xlabel(r"$\beta$", size=14)
    ax.set_ylabel(r"$\sigma$", size=14)
    ax.set_xticks(betas)
axs[0].set_title("Identity sensitivity", size=16)
axs[1].set_title("Neighbourhood sensitivity", size=16)

# Create a colorbar that aligns perfectly
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Create an axis for the colorbar
divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.05)  # Adjust size and padding
cbar = plt.colorbar(im0, cax=cax)  # Use the new axis for the colorbar
# Same for the second one
divider = make_axes_locatable(axs[1])
cax = divider.append_axes("right", size="5%", pad=0.05)  # Adjust size and padding
cbar = plt.colorbar(im1, cax=cax)  # Use the new axis for the colorbar

fig.tight_layout()

if SAVEFIG:
    # Note: saving as PDF fails to show the hatches
    plt.savefig(f"ID-and-NS-res3.png", dpi=600)

## Comparison with Lyapunov spectrum

Each of the local update rules can be compared to their Lyapunov spectrum. That is a good indicator of how good a measure this is.

First, let us consider the idea that every totalistic ECA can be translated to a LLNA on a regular graph. In particular, such ECAs can be encoded in resolution-3 LLNAs. Note that there are $2^4$ totalistic ECAs, and $2^6$ resolution-3 LLNAs, so this mapping is not bijective.

An ECA is totalistic if the order of the cells does not matter. This means that a binary representation of the ECA should be $b_7 b_6 b_5 b_4 b_3 b_2 b_1 b_0$ with
$$
\begin{align}
\begin{cases}
b_6 = b_5 = b_3, \qquad &\text{(two living cells)}\\
b_4 = b_2 = b_1. \qquad &\text{(one living cells)}
\end{cases}
\end{align}
$$

Actually, there is a bijective map from ECA to LLNA, by demanding that
$$
\begin{align}
\begin{cases}
b_6 = b_3, \qquad &\text{(alive, one neighbour)}\\
b_4 = b_1. \qquad &\text{(dead, one neighbour)}
\end{cases}
\end{align}
$$

This indeed sums to $64$ possibilities.

Now we are especially interested in the ECAs that have a constant Jacobian, meaning that its response to a defect does not depend on the particular state configuration. Here the demand is that the value of the gradient does _not_ depend on the configuration, i.e.
$$
\left(\frac{\partial f}{\partial x_{i-1}}, \frac{\partial f}{\partial x_{i}}, \frac{\partial f}{\partial x_{i+1}}\right) \in \{0,1\}^3,
$$
where
$$
\frac{\partial f}{\partial x_{i}} = f(x_{i-1}, x_i, x_{i+1}) \oplus f(x_{i-1}, \bar{x}_i, x_{i+1}).
$$
Here $f$ is the local update rule of the ECA, and the $\bar{x}$ denotes the binary complement ($\bar{x} = 1-x$). This means that we must demand that
$$
\begin{align}
\begin{cases}
f(1, 1, 1) \oplus f(1, 0, 1) = k \in \{0, 1\}, \\
f(1, 1, 0) \oplus f(1, 0, 0) = k, \\
f(0, 1, 1) \oplus f(0, 0, 1) = k, \\
f(0, 1, 0) \oplus f(0, 0, 0) = k.
\end{cases}
\end{align}
$$
In binary notation, this becomes
$$
\begin{align}
\begin{cases}
b_7 \oplus b_5 = k \in \{0, 1\}, \\
b_6 \oplus b_4 = k, \\
b_3 \oplus b_1 = k, \\
b_2 \oplus b_0 = k.
\end{cases}
\end{align}
$$
This in turn translates to the following demand:
$$
\begin{align}
[ (b_7 = b_5) \land (b_6 = b_4) \land (b_3 = b_1) \land (b_2 = b_0) ] \lor [ (b_7 = \bar{b}_5) \land (b_6 = \bar{b}_4) \land (b_3 = \bar{b}_1) \land (b_2 = \bar{b}_0) ].
\end{align}
$$

TODO: this is not correct! I'm missing something; I'm counting twice as many as I need.

In [ ]:
from src.analysis import eca_is_llna, eca_has_constantJ, eca_to_llna, lyapunov_spectrum_analytical
constantJ_llnas = [eca for eca in range(256) if (eca_is_llna(eca) and eca_has_constantJ(eca))]

# hardcoded
# TODO: this can be automated by checking the equivalent ECAs
nonequiv_constantJ_llnas = [0, 51, 204, 90, 105, 150]

# find singular values analytically
multiplier=20
N = 2*3*4*multiplier+1 # 2*3*4*20+1
bins = np.linspace(-4, np.log(3), 20)

fig, axs = plt.subplots(len(nonequiv_constantJ_llnas), 2, figsize=(7, 7), width_ratios=[4,1])

resolution=3
for ax, eca in zip(axs, nonequiv_constantJ_llnas):
    ax_left, ax_right = ax
    # make LLNA diagram
    born_if, survive_if = eca_to_llna(eca)
    model = LLNA(resolution, x=born_if, y=survive_if, iso=True)
    model.diagram(ax=ax_left)
    IS = id_sensitivity(resolution, born_if, survive_if)
    NS = nbh_sensitivity(resolution, born_if, survive_if)
    ax_left.set_title(fr"ECA {eca} (LLNA {model.__str__(latex=True)}). $\text{{IS}}={IS}$, $\text{{NS}}={NS}$")
    ax_left.set_ylabel(None)
    # make Lyapunov spectrum. Takes a little while to calculate
    lyapunov_values, finite_pct = lyapunov_spectrum_analytical(eca, N, return_finite_pct=True)
    ax_right.hist(lyapunov_values, bins=bins)
for ax in axs[:-1]:
    ax_left, ax_right = ax
    ax_left.set_xlabel(None)
    # ax_left.set_xticks([])
    ax_left.get_legend().remove()
    ax_right.set_ylim([0,N*1.05])
    ax_right.set_yticks([])
    ax_right.set_xticks([0])

axs[0,1].set_title(f"Lyapunov spectrum", size=12)
axs[-1,0].set_xlabel(fr"State density $\rho_j$ associated with node $v_j$", size=12)
axs[-1,1].set_ylim([0,N*1.05])
axs[-1,1].set_yticks([])
axs[-1,1].set_xticks([0])

# clean and save
fig.supylabel("Node becomes ...", size=14)
fig.suptitle("Local update rule diagrams for non-equivalent constant-Jacobian ECAs", size=14)
fig.tight_layout()

if SAVEFIG:
    # Note: saving as PDF fails to show the hatches
    plt.savefig(f"simple-nonequiv-llnas-vs-analytical-lyapunov-spectrum.png", dpi=600)

## Comparison with global damage density evolution

Below we look at all $64$ LLNAs that are equivalent to ECAs. First we filter out the non-equivalent ones. Next we look at the evolution of the damage density in the entire system.

In [ ]:
from src.analysis import lr_symmetric_eca, bw_symmetric_eca, eca_is_llna, eca_to_llna

# find array of non-equivalent ECAs that can be described as an LLNA
ecas_equiv_to_llna = []
for eca in range(256):
    if eca_is_llna(eca):
        lr_eca = lr_symmetric_eca(eca)
        bw_eca = bw_symmetric_eca(eca)
        if (lr_eca not in ecas_equiv_to_llna) and (bw_eca not in ecas_equiv_to_llna):
            # note that left-right inversion has no effect on ECAs that are described by LLNAs
            ecas_equiv_to_llna.append(eca)
ecas_equiv_to_llna = np.array(ecas_equiv_to_llna)

# turn these ECAs into LLNAs
models = []
IS_values = []
NS_values = []
for eca in ecas_equiv_to_llna:
    born_if, survive_if = eca_to_llna(eca)
    model = LLNA(3, x=born_if, y=survive_if, iso=True)
    IS = id_sensitivity(resolution, born_if, survive_if)
    NS = nbh_sensitivity(resolution, born_if, survive_if)
    models.append(model)
    IS_values.append(IS)
    NS_values.append(NS)

In [ ]:
# show an example

randint = np.random.randint(len(ecas_equiv_to_llna))

eca_equiv_to_llna = ecas_equiv_to_llna[randint]

# make a regular graph (emulating ECAs)
multiplier = 2
N = 2*3*4*multiplier+1
G_ring = ig.Graph.Ring(N)
# get ID of edges (bidirectional)
G_ring.to_directed()
edges = tc.tensor(G_ring.get_edgelist()).T
G_ring.to_undirected()

T = N//2
# states = tc.tensor(np.random.randint(2, size=(1,N)))
# states_seed = np.zeros(N)
# states_seed[N//2] = 1
states_seed = np.random.randint(2,size=N)
H = models[randint].forward(edges, tc.tensor(states_seed[np.newaxis,:]), T=T)
H = np.array(H, dtype=int)

fig, axs = plt.subplots(1,2,figsize=(10,4))
ig.plot(G_ring,target=axs[0], vertex_size=10)
axs[1].imshow(H[0], cmap='Greys')
axs[1].set_title(f"Rule {eca_equiv_to_llna} emulated in a ring network")
axs[1].set_xlabel("Nodes in the ring network")
axs[1].set_ylabel("Time steps")

Now we want to show what happens to the global state density of the difference patter (i.e. the fraction of defect nodes) over time. First, let's consider a single one.

In [ ]:
# show an example
randint = np.random.randint(len(ecas_equiv_to_llna))

eca_equiv_to_llna = ecas_equiv_to_llna[randint]
model = models[randint]

# make a regular graph (emulating ECAs)
multiplier = 2
N = 2*3*4*multiplier+1
G_ring = ig.Graph.Ring(N)
# get ID of edges (bidirectional)
G_ring.to_directed()
edges = tc.tensor(G_ring.get_edgelist()).T
G_ring.to_undirected()

# random initial configuration and one with the defect
states_seed = np.random.randint(2,size=N)
states_seed_defect = states_seed.copy()
states_seed_defect[N//2] = 1-states_seed_defect[N//2]

# evolve over T time steps
T = N//2
H = model.forward(edges, tc.tensor(states_seed[np.newaxis,:]), T=T)
H = np.array(H, dtype=int)
H_defect = model.forward(edges, tc.tensor(states_seed_defect[np.newaxis,:]), T=T)
H_defect = np.array(H_defect, dtype=int)

# find difference pattern and its mean over time
H_difference = np.bitwise_xor(H, H_defect)
H_difference_dens = np.mean(H_difference[0], axis=1)

fig, axs = plt.subplots(1,2,figsize=(10,3.5))

axs[0].imshow(H[0], cmap='Greys')
axs[0].imshow(H_defect[0], cmap='Greys', alpha=.4)
axs[0].set_title(fr"ECA {eca_equiv_to_llna}, LLNA {model.__str__(latex=True)}", size=14)
axs[0].set_xticks([])
axs[0].set_yticks([])

axs[1].plot(H_difference_dens)
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Global difference pattern density")

fig.tight_layout()

Now average this over many initial configurations.

In [ ]:
# show an example
randint = np.random.randint(len(ecas_equiv_to_llna))
N_samples = 10

eca_equiv_to_llna = ecas_equiv_to_llna[randint]
model = models[randint]
IS = IS_values[randint]
NS = NS_values[randint]

# make a regular graph (emulating ECAs)
multiplier = 2
N = 2*3*4*multiplier+1
G_ring = ig.Graph.Ring(N)
# get ID of edges (bidirectional)
G_ring.to_directed()
edges = tc.tensor(G_ring.get_edgelist()).T
G_ring.to_undirected()

H_difference_dens_list = []
for _ in range(N_samples):
    # random initial configuration and one with the defect
    states_seed = np.random.randint(2,size=N)
    states_seed_defect = states_seed.copy()
    states_seed_defect[N//2] = 1-states_seed_defect[N//2]

    # evolve over T time steps
    T = N//2
    H = model.forward(edges, tc.tensor(states_seed[np.newaxis,:]), T=T)
    H = np.array(H, dtype=int)
    H_defect = model.forward(edges, tc.tensor(states_seed_defect[np.newaxis,:]), T=T)
    H_defect = np.array(H_defect, dtype=int)

    # find difference pattern and its mean over time
    H_difference = np.bitwise_xor(H, H_defect)
    H_difference_dens = np.mean(H_difference[0], axis=1)
    H_difference_dens_list.append(H_difference_dens)
H_difference_dens_list = np.array(H_difference_dens_list)

# # plot
fig, axs = plt.subplots(1,2,figsize=(10,3.3))
size=14

axs[0].imshow(H[0], cmap='Greys')
axs[0].imshow(H_defect[0], cmap='Greys', alpha=.4)
axs[0].set_title(fr"ECA {eca_equiv_to_llna}, LLNA {model.__str__(latex=True)}, IS={round(IS,2)}, NS={round(NS,2)}", size=14)
axs[0].set_xticks([])
axs[0].set_yticks([])

H_means = np.mean(H_difference_dens_list, axis=0)
H_sigma = np.std(H_difference_dens_list, axis=0)

axs[1].plot(H_difference_dens_list.T, color='k', alpha=2*1/N_samples)
axs[1].plot(H_means, color='k', label='Mean defect density')
axs[1].fill_between(range(T+1), H_means-H_sigma, H_means+H_sigma, color='k', alpha=0.3, label='Standard deviation')
axs[1].legend(fontsize=size)
axs[1].set_xlabel("Time", size=size)
axs[1].set_ylabel("Global difference pattern density", size=size)
axs[1].set_ylim([0,.7])

aspect=H[0].shape[0]/H[0].shape[1]
axs[0].set_box_aspect(aspect)  # Keeps aspect ratio controlled
axs[1].set_box_aspect(aspect)

fig.tight_layout()

Let us, as an intermediate step, take a brief look at the relationship between the Li-Packard class and the sensitivity measures defined in this Notebook. We would expect that the null, fixed-point and periodic classes have low sensitivity values, and that (locally) chaotic classes have high sensitivity. It's not that straightforward, however, as is seen below.

In [ ]:
# show scatter plot of LP classes versus IS and NS values
from src.analysis import lp_class_dict

# fetch Li-Packard dictionary
lp_dict = lp_class_dict()
lp_dict.keys()

# invert lp_dict
lp_dict_inv = {}
for key, values in lp_dict.items():
    for value in values:
        lp_dict_inv[value] = key  # Assign integer as key and string as value

# find LP class of each non-equivalent ECA that is isomorphic to an ECA
lp_classes_equiv_to_llna = np.array([lp_dict_inv[eca] for eca in ecas_equiv_to_llna])

# create color coding
lp_class_to_color = dict({'null' : 'black',
                          'fixed point' : 'red',
                          'periodic' : 'blue',
                          'locally chaotic' : 'green',
                          'chaotic' : 'orange'})
color = np.array([lp_class_to_color[lp_class] for lp_class in lp_classes_equiv_to_llna])

fig, ax = plt.subplots(1,1,figsize=(6,4))
size=14

# jitter for visibility
jitter_size = 0.05
h_jitter = (np.random.rand(len(IS_values))-.5)*jitter_size
v_jitter = (np.random.rand(len(IS_values))-.5)*jitter_size
ax.scatter(IS_values+h_jitter, NS_values+v_jitter, color=color, alpha=0.6)

# Custom legend using patches
import matplotlib.lines as mlines
legend_handles = [mlines.Line2D([0], [0], linestyle='None', marker='o', markersize=6, color=color, label=label)
                  for label, color in lp_class_to_color.items()]
ax.legend(handles=legend_handles, ncols=3)
ax.set_title("NS vs. IS vs. Li-Packard classification\nfor all 36 non-equivalent ECA described as LLNAs", size=size)
ax.set_xlabel("Identity Sensitivity", size=size)
ax.set_ylabel("Neighbourhood Sensitivity", size=size)

fig.tight_layout()

lp_classes, counts = np.unique(lp_classes_equiv_to_llna, return_counts=True)
for lp_class, count in zip(lp_classes, counts):
    print(f"Class '{lp_class}': {count}/36 occurences.")

All (locally) chaotic ECAs have a NS of 0.5 or higher. IS appears to be a bad predictor.

Now let us consider various definitions of useful metrics for ECAs. Various possibilities present themselves
1. Width of defect. This is equivalent to the diameter of the affected subnetwork. This should probably be normalised over time in some sense.
2. Size of defect. This is the number of defect cells. This should probably be normalised over the width and over the time.
3. Variance of the values from the metrics indicated above.

In [ ]:
# NOTE: the diameter of the affected subgraph has been studied in the Overview file before

def defect_subgraph(graph, deltas):
    """
    TODO: make this nicer
    this function takes as inputs:
        - the defect pattern over time
        - information of the network structure (the list of edges)
    this function outputs
        - the subgraph consisting of all nodes that have been defected at least once
    """
    # find cumulative defect pattern
    deltas_cum = np.cumsum(deltas, axis=0)
    deltas_cum = np.clip(deltas_cum, 0, 1)[-1]
    # 
    subG = graph.subgraph(np.where(deltas_cum)[0])
    return subG

# make a regular graph (emulating ECAs)
multiplier = 2
N = 2*3*4*multiplier+1
G_ring = ig.Graph.Ring(N)
# get ID of edges (bidirectional)
G_ring.to_directed()
edges = tc.tensor(G_ring.get_edgelist()).T
G_ring.to_undirected()

# number of samples
N_samples = 8

subgraph_diameter_1Qs = []
subgraph_diameter_2Qs = []
subgraph_diameter_3Qs = []
# use list of models defined earlier
for i, eca_equiv_to_llna in enumerate(ecas_equiv_to_llna):
    model = models[i]

    subgraph_diameters = []
    for _ in range(N_samples):
        # random initial configuration and one with the defect
        states_seed = np.random.randint(2,size=N)
        states_seed_defect = states_seed.copy()
        states_seed_defect[N//2] = 1-states_seed_defect[N//2]

        # evolve over T time steps
        T = N//2-1
        H = model.forward(edges, tc.tensor(states_seed[np.newaxis,:]), T=T)
        H = np.array(H, dtype=int)
        H_defect = model.forward(edges, tc.tensor(states_seed_defect[np.newaxis,:]), T=T)
        H_defect = np.array(H_defect, dtype=int)

        # find difference pattern
        H_difference = np.bitwise_xor(H, H_defect)
        deltas = H_difference[0]

        # find subgraph and its diameter
        subG = defect_subgraph(G_ring, deltas)
        subgraph_diameter = subG.diameter()

        # add to list
        subgraph_diameters.append(subgraph_diameter)
    subgraph_diameters = np.array(subgraph_diameters)
    subgraph_diameter_1Q = np.quantile(subgraph_diameters, .25)
    subgraph_diameter_2Q = np.quantile(subgraph_diameters, .5)
    subgraph_diameter_3Q = np.quantile(subgraph_diameters, .75)

    subgraph_diameter_1Qs.append(subgraph_diameter_1Q)
    subgraph_diameter_2Qs.append(subgraph_diameter_2Q)
    subgraph_diameter_3Qs.append(subgraph_diameter_3Q)
        

In [ ]:
yerr = [np.array(subgraph_diameter_2Qs) - np.array(subgraph_diameter_1Qs),
        np.array(subgraph_diameter_3Qs) - np.array(subgraph_diameter_2Qs)]
x_jitter_size=0.02
x_jitter = (np.random.random(len(NS_values))-.5)*x_jitter_size

fig, axs = plt.subplots(1,2,figsize=(10,3.5))

axs[0].errorbar(np.array(IS_values)+x_jitter, subgraph_diameter_2Qs, yerr=yerr, fmt='o', capsize=5, capthick=1, elinewidth=1, color='darkgreen')
axs[1].errorbar(np.array(NS_values)+x_jitter, subgraph_diameter_2Qs, yerr=yerr, fmt='o', capsize=5, capthick=1, elinewidth=1, color='maroon')

axs[0].set_title("Defect diameter vs. identity sensitivity")
axs[1].set_title("Defect diameter vs. neighbourhood sensitivity")

axs[0].set_ylabel("Defect diameter")
axs[1].set_ylabel("Defect diameter")
axs[0].set_xlabel("IS value")
axs[1].set_xlabel("NS value")

fig.suptitle(f"All {len(NS_values)} LLNAs that correspond to ECAs. {N_samples} samples, {N} cells, {T} time steps", fontsize=16)
fig.tight_layout()

Now we generalise this to all possible LLNAs on a regular graph. After that, we consider more general graphs.

In [ ]:
from src.analysis import defect_diameter
import itertools

# make a regular graph (emulating ECAs)
multiplier = 10
N = 2*3*4*multiplier+1
G_ring = ig.Graph.Ring(N)

# make the set of all possible S_sets and B_sets
resolution = 3 # a resolution higher than 3 does not make sense for a regular graph!
R_set = range(resolution)
power_set = list(itertools.chain.from_iterable(
    itertools.combinations(R_set, r) for r in range(len(R_set) + 1)
))

diams_per_model = []
IS_per_model = []
NS_per_model = []
states=np.random.randint(2,size=N)
T=N//2-1
for i, B_set in enumerate(power_set):
    for j, S_set in enumerate(power_set):
        # print to user
        print(f"Working on B_set {i+1}/{2**resolution}, S_set {j+1}/{2**resolution}.  ", end='\r')
        # find defect diameter
        model = LLNA(resolution, x=B_set, y=S_set)
        diams = defect_diameter(G_ring, model, states=states, T=T, norm=False)
        diams_per_model.append(diams)
        # find IS and NS
        IS = id_sensitivity(resolution, B_set, S_set)
        NS = nbh_sensitivity(resolution, B_set, S_set)
        IS_per_model.append(IS)
        NS_per_model.append(NS)

diams_per_model = np.array(diams_per_model)

In [ ]:
diams_per_model_1Q = np.quantile(diams_per_model, .25, axis=1)
diams_per_model_2Q = np.quantile(diams_per_model, .50, axis=1)
diams_per_model_3Q = np.quantile(diams_per_model, .75, axis=1)

yerr = [np.array(diams_per_model_2Q) - np.array(diams_per_model_1Q),
        np.array(diams_per_model_3Q) - np.array(diams_per_model_2Q)]
x_jitter_size=0.02
x_jitter = (np.random.random(len(NS_per_model))-.5)*x_jitter_size

fig, axs = plt.subplots(1,2,figsize=(10,3.5))

axs[0].errorbar(np.array(IS_per_model)+x_jitter, diams_per_model_2Q, yerr=yerr, fmt='o', capsize=5, capthick=1, elinewidth=1, color='darkgreen')
axs[1].errorbar(np.array(NS_per_model)+x_jitter, diams_per_model_2Q, yerr=yerr, fmt='o', capsize=5, capthick=1, elinewidth=1, color='maroon')

axs[0].set_title("Defect diameter vs. identity sensitivity")
axs[1].set_title("Defect diameter vs. neighbourhood sensitivity")

axs[0].set_ylabel("Defect diameter")
axs[1].set_ylabel("Defect diameter")
axs[0].set_xlabel("IS value")
axs[1].set_xlabel("NS value")

fig.suptitle(f"All {2**(2*resolution)} LLNAs of resolution {resolution} on a ring network. {N} samples, {N} cells, {T} time steps", fontsize=16)
fig.tight_layout()

We can increase the resolution, but that only makes sense if we simultaneously allow for larger neighbourhoods. Let us consider a ring network with degree $4$.

In [ ]:
# make a regular graph (emulating ECAs)
multiplier = 1
N = 2*3*4*multiplier+1
G_regular = ig.Graph(N)

# Add undirected edges to form a ring (nearest neighbors) and add second-nearest neighbors
G_regular.add_edges([(i, (i + 1) % N) for i in range(N)])  # First neighbours
G_regular.add_edges([(i, (i + 2) % N) for i in range(N)])  # Second neighours

fig, axs = plt.subplots(1,2,figsize=(8,4))

ig.plot(G_regular, target=axs[0], vertex_size=10)
axs[1].imshow(np.array(G_regular.get_adjacency()), cmap='Greys')

axs[0].set_title(f"Network topology (N={N})")
axs[1].set_title("Network adjacency matrix")

fig.tight_layout()

In [ ]:
from src.analysis import defect_diameter
import itertools

# make a regular graph (emulating ECAs)
multiplier = 5
N = 2*3*4*multiplier+1
G_regular = ig.Graph(N)

# Add undirected edges to form a ring (nearest neighbors) and add second-nearest neighbors
G_regular.add_edges([(i, (i + 1) % N) for i in range(N)])  # First neighbours
G_regular.add_edges([(i, (i + 2) % N) for i in range(N)])  # Second neighours

# make the set of all possible S_sets and B_sets
resolution = 5 # a resolution higher than k+1 does not make sense for a ring graph with degree k!
R_set = range(resolution)
power_set = list(itertools.chain.from_iterable(
    itertools.combinations(R_set, r) for r in range(len(R_set) + 1)
))

diams_per_model = []
IS_per_model = []
NS_per_model = []
states=np.random.randint(2,size=N)
T=N//4-1
for i, B_set in enumerate(power_set):
    for j, S_set in enumerate(power_set):
        # print to user
        print(f"Working on B_set {i+1}/{2**resolution}, S_set {j+1}/{2**resolution}.  ", end='\r')
        # find defect diameter
        model = LLNA(resolution, x=B_set, y=S_set)
        diams = defect_diameter(G_regular, model, states=states, T=T, norm=False)
        diams_per_model.append(diams)
        # find IS and NS
        IS = id_sensitivity(resolution, B_set, S_set)
        NS = nbh_sensitivity(resolution, B_set, S_set)
        IS_per_model.append(IS)
        NS_per_model.append(NS)

diams_per_model = np.array(diams_per_model)

In [ ]:
diams_per_model_1Q = np.quantile(diams_per_model, .25, axis=1)
diams_per_model_2Q = np.quantile(diams_per_model, .50, axis=1)
diams_per_model_3Q = np.quantile(diams_per_model, .75, axis=1)

yerr = [np.array(diams_per_model_2Q) - np.array(diams_per_model_1Q),
        np.array(diams_per_model_3Q) - np.array(diams_per_model_2Q)]
x_jitter_size=0.03
x_jitter = (np.random.random(len(NS_per_model))-.5)*x_jitter_size

fig, ax = plt.subplots(1,1,figsize=(10,3.5))

sc = ax.scatter(np.array(NS_per_model)+x_jitter, diams_per_model_2Q, c=IS_per_model, cmap='summer', s=10, alpha=1)
# Add colorbar as legend
cbar = plt.colorbar(sc)

fontsize=14
cbar.set_label("Identity sensitivity", fontsize=fontsize)
ax.set_xlabel("Neighbourhood sensitivity", fontsize=fontsize)
ax.set_ylabel("Diameter of\naffected subgraph", fontsize=fontsize)

fig.suptitle(f"All {2**(2*resolution)} LLNAs of resolution {resolution} on a ring network with degree 4.\n{N} samples, {N} cells, {T} time steps.", fontsize=16)
fig.tight_layout()

It's not entirely clear what we should take away from this. Some observations:
1. There is a trend, a clear correlation between NS and diameter of the affected subgraph
2. For intermediate values of the NS, it is not obvious what the diameter should be
3. The IS seems to not contain additional information that helps to distinguish between 'upper' and 'lower' regimes
4. There seems to be a gap in the middle: either the network is barely affected, either it is almost entirely affected.

I should check the relationship with the **Langton parameter**!

I was hoping for a more clear picture, here. This is a very simple topology, yet the combination of NS and IS is not capable of clearly predicting the outcome of the LLNA. When we go towards non-trivial topologies, this will not magically become better. What makes it extra annoying is that the region with the most ambiguity is also the region with the highest number of LLNA rules: there are many rules with NS around 0.5, and only few with NS around 0 or 1.

In [ ]:
# TODO: use machine learning to find the predictive capacity of Is and NS regarding LP class
# TODO: question: what is the difference between IS and NS. Belangrijk om dit voldoende te snappen: kijken wat het is dat IS zegt en wat NS zegt, welke andere informatie het in zich draagt
# TODO: neem een goeie metriek voor het kwantificeren van de damage spread. Bijvoorbeeld ook kijken naar de jigginess van een lijn
# TODO: het allermooiste zou zijn om twee metrieken te halen uit de curves rechts en te kijken hoe die gerelateerd zijn aan de IS en NS
# TODO: lees manuscript van Lucas

# NOTE: misschien zegt IS niks over chaoticiteit, maar het kan wel nuttig zijn om accuracy the voorspellen
# NOTE: het systeem is inderdaad weinig afhankelijk van IS, zeker in netwerken met een hoge gemiddelde degree
# TODO: ik heb het gevoel dat een LLNA waarvan de node zelf ook in de neighbourhood zit, altijd kan herschreven worden naar een LLNA waarbij dat niet het geval is (maar omgekeerd niet). Dat kan ik misschien wel eens formaliseren. Wellicht handig om dat mee in de basisdefinitie te steken. DIT IS VAN BELANG VOOR EEN LINK MET CAs! Meer precies: "every totalistic CA can be phrased as an outer-totalitic CA". Misschien kan ik deze equivalence gebruiken om een goede definitie van sensitivity te verantwoorden
# TODO: kan ik een goed argument maken waarom dat LLNAs met een even resolutie geen steek houden? We zitten dan met een niet-uniforme breedte van de densiteitsintervals. Maar de onwenselijkheid daarvan moet beter verantwoord worden.
# NOTE: another reason to prefer large 'jaggedness' is that low jaggedness has a lot in common with just low resolution. (bias)
# NOTE: jaggedness is also just 'sensitivity to change', which is also what the Jacobian expresses. Can I formalise this for ECAs? That would help to find a better foundation for the metric

# TODO: jaggedness is CLEARLY related to what's known as the mu-sensitivity (which in turn is related to the Lyapunov spectrum). Make sure not to miss this point.